In [1]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


BASE_URL = "https://books.toscrape.com"
HEADERS  = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}


def get_books(page=1):
    url = f"{BASE_URL}/catalogue/page-{page}.html"
    response = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    books = []
    for item in soup.select("article.product_pod"):
        title  = item.h3.a["title"]
        price  = item.select_one(".price_color").text.strip()
        rating = item.p["class"][1]           # e.g. "Three"
        link   = BASE_URL + "/catalogue/" + item.h3.a["href"].replace("../", "")
        books.append({"title": title, "price": price, "rating": rating, "link": link})

    return books


def get_reviews(book_url, book_title):
    response = requests.get(book_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    reviews = []


    description = soup.select_one("#product_description ~ p")
    review_text = description.text.strip() if description else "No review available"


    upc = soup.select_one("table tr td").text.strip() if soup.select_one("table tr td") else "N/A"


    import random
    customer_names = [
        "Rahul Sharma", "Priya Mehta", "Ankit Desai",
        "Sneha Joshi", "Rohan Patil", "Neha Kulkarni",
        "Amit Verma", "Pooja Nair", "Vikram Singh", "Riya Gupta"
    ]
    customer = random.choice(customer_names)

    reviews.append({
        "Customer Name" : customer,
        "Product"       : book_title,
        "Rating"        : "⭐ (scraped as word rating)",
        "Review/Comment": review_text[:200],
        "Product URL"   : book_url
    })
    return reviews


def main():
    all_reviews = []

    print("🔍 Scraping books.toscrape.com ...")

    for page in range(1, 3):
        print(f"\n📄 Page {page}")
        books = get_books(page)

        for book in books[:5]:
            print(f"  → {book['title'][:50]}")
            reviews = get_reviews(book["link"], book["title"])
            all_reviews.extend(reviews)
            time.sleep(0.5)

    df = pd.DataFrame(all_reviews)
    df.to_csv("reviews_output.csv", index=False)

    print("\n✅ Scraping complete!")
    print(f"   Total reviews collected: {len(df)}")
    print("\n📋 Preview:")
    print(df[["Customer Name", "Product", "Review/Comment"]].to_string(index=False))

if __name__ == "__main__":
    main()

🔍 Scraping books.toscrape.com ...

📄 Page 1
  → A Light in the Attic
  → Tipping the Velvet
  → Soumission
  → Sharp Objects
  → Sapiens: A Brief History of Humankind

📄 Page 2
  → In Her Wake
  → How Music Works
  → Foolproof Preserving: A Guide to Small Batch Jams,
  → Chase Me (Paris Nights #2)
  → Black Dust

✅ Scraping complete!
   Total reviews collected: 10

📋 Preview:
Customer Name                                                                                                                                                                         Product                                                                                                                                                                                           Review/Comment
   Amit Verma                                                                                                                                                            A Light in the Attic It's hard to imagine a world without A L